In [2]:
# 셀 1: 패키지 설치
!pip install langchain langchain-openai langchain-community langchain-experimental faiss-cpu python-dotenv

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_experimental-0.4.0-py3-none-any.whl.metadata (1.3 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.0.2-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_openai-1.0.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_openai-1.0.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_openai-0.3.35-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_core-0.3.79-py3-none-any.whl.metadata (3.2 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community

In [3]:
pip install langchain langchain-community langchain-core faiss-cpu sentence-transformers rank-bm25 konlpy numpy scikit-learn python-dotenv tqdm

  Using cached sentence_transformers-5.1.2-py3-none-any.whl.metadata (16 kB)
  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached sentence_transformers-5.1.2-py3-none-any.whl (488 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
Using cached hf_xet-1.2.0-cp37-abi3-macosx_11_0_arm64.whl (2.7 MB)
Using cached tokenizers-0

In [4]:
# 필요한 라이브러리 설치 (처음 한 번만)
# !pip install langchain langchain-openai langchain-community langchain-experimental faiss-cpu python-dotenv

import os
import json
import re
from typing import List, Dict, Any
from datetime import datetime

from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_experimental.text_splitter import SemanticChunker
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

print(" 라이브러리 import 완료!")

 라이브러리 import 완료!


## config

### Settings.py

## Data

### load_data.py

In [ ]:
# ========================================
# 데이터 소스 정의 (전체)
# ========================================
print("=" * 70)
print("데이터 소스 정의")
print("=" * 70)

data_sources = [
    # ========= 기존 데이터 =========
    {
        "path": "/Users/jinwoong/Desktop/coding/Bitamin/nlp_project_2/dataset/dacon/dacon.json",
        "type": "competition",
        "platform": "dacon",
        "description": "데이콘 AI 경진대회"
    },
    {
        "path": "/Users/jinwoong/Desktop/coding/Bitamin/nlp_project_2/dataset/Inflearn/inflearn_courses_all.json",
        "type": "education",
        "platform": "inflearn",
        "description": "인프런 온라인 강의"
    },
    
    # ========= 새로 추가 =========
    {
        "path": "/Users/jinwoong/Desktop/coding/Bitamin/nlp_project_2/dataset/lh_compas/lh_compas_산학협력.json",
        "type": "competition",
        "platform": "lh_compas",
        "description": "LH 컴퍼스 산학협력"
    },
    {
        "path": "/Users/jinwoong/Desktop/coding/Bitamin/nlp_project_2/dataset/lh_compas/lh_compas_아이디어공모전.json",
        "type": "competition",
        "platform": "lh_compas",
        "description": "LH 컴퍼스 아이디어 공모전"
    },
    {
        "path": "/Users/jinwoong/Desktop/coding/Bitamin/nlp_project_2/dataset/kaggle/kaggle_active_korean.json",
        "type": "competition",
        "platform": "kaggle",
        "description": "Kaggle 경진대회 (한국어)"
    },
    {
        "path": "/Users/jinwoong/Desktop/coding/Bitamin/nlp_project_2/dataset/linkareer/linkareer.json",
        "type": "auto",  # 자동 판단
        "platform": "linkareer",
        "description": "링커리어 (대외활동/공모전 자동 구분)"
    },
    {
        "path": "/Users/jinwoong/Desktop/coding/Bitamin/nlp_project_2/dataset/공공데이터포털/data_go_kr.json",
        "type": "competition",
        "platform": "data_go_kr",
        "description": "공공데이터포털 공모전"
    },
]

print(f"\n총 {len(data_sources)}개 데이터 소스")
for i, source in enumerate(data_sources, 1):
    print(f"  [{i}] {source['platform']:15s} - {source['description']}")

print("=" * 70)

데이터 소스 정의

총 7개 데이터 소스
  [1] dacon           - 데이콘 AI 경진대회
  [2] inflearn        - 인프런 온라인 강의
  [3] lh_compas       - LH 컴퍼스 산학협력
  [4] lh_compas       - LH 컴퍼스 아이디어 공모전
  [5] kaggle          - Kaggle 경진대회 (한국어)
  [6] linkareer       - 링커리어 (대외활동/공모전 자동 구분)
  [7] data_go_kr      - 공공데이터포털 공모전


In [ ]:
# ========================================
# 전체 데이터 로드
# ========================================
print("=" * 70)
print("전체 데이터 로드")
print("=" * 70)

import json
from langchain.schema import Document

all_documents = []

for source in data_sources:
    platform = source['platform']
    data_type = source['type']
    file_path = source['path']
    
    print(f"\n {platform} 로드 중...")
    
    try:
        # JSON 로드
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        
        print(f"   원본: {len(raw_data)}개")
        
        # 데이터 처리
        for item in raw_data:
            title = item.get('title', '제목 없음')
            content = item.get('content', item.get('description', ''))
            url = item.get('url', '')
            
            # linkareer는 자동 구분
            if platform == 'linkareer' and data_type == 'auto':
                doc_type = classify_linkareer_type(item)
            else:
                doc_type = data_type
            
            # Document 생성
            doc = Document(
                page_content=f"제목: {title}\n\n{content}",
                metadata={
                    'title': title,
                    'type': doc_type,
                    'platform': platform,
                    'url': url,
                }
            )
            all_documents.append(doc)
        
        print(f"    {len(raw_data)}개 완료")
        
    except FileNotFoundError:
        print(f"    파일 없음: {file_path}")
    except Exception as e:
        print(f"    오류: {e}")

print("\n" + "=" * 70)
print(f" 총 {len(all_documents)}개 문서 로드")

# 통계
from collections import Counter
platform_counts = Counter([d.metadata['platform'] for d in all_documents])
type_counts = Counter([d.metadata['type'] for d in all_documents])

print("\n 플랫폼별:")
for p, c in platform_counts.most_common():
    print(f"   {p:15s}: {c:4d}개")

print("\n 타입별:")
for t, c in type_counts.most_common():
    print(f"   {t:15s}: {c:4d}개")

print("=" * 70)

전체 데이터 로드

 dacon 로드 중...
   원본: 243개
    243개 완료

 inflearn 로드 중...
   원본: 7502개
    7502개 완료

 lh_compas 로드 중...
   원본: 6개
    6개 완료

 lh_compas 로드 중...
   원본: 42개
    42개 완료

 kaggle 로드 중...
   원본: 20개
    20개 완료

 linkareer 로드 중...
   원본: 20개
    20개 완료

 data_go_kr 로드 중...
   원본: 94개
    94개 완료

 총 7927개 문서 로드

 플랫폼별:
   inflearn       : 7502개
   dacon          :  243개
   data_go_kr     :   94개
   lh_compas      :   48개
   kaggle         :   20개
   linkareer      :   20개

 타입별:
   education      : 7502개
   competition    :  425개


### preprocess.py

In [ ]:
#단일 문서 document전환 함수
def create_document(item: Dict, doc_type: str, platform: str) -> Document:
    """JSON 항목을 LangChain Document로 변환"""
    
    # 1. 텍스트 내용 구성 (null 처리)
    parts = []
    
    # 제목 (필수)
    if item.get('제목'):
        parts.append(f"제목: {item['제목']}")
    
    # 공모전인 경우
    if doc_type == 'competition':
        if item.get('배경'):
            parts.append(f"배경: {item['배경']}")
        if item.get('주제'):
            parts.append(f"주제: {item['주제']}")
        if item.get('설명'):
            parts.append(f"설명: {item['설명']}")
        if item.get('참가 대상'):
            parts.append(f"참가 대상: {item['참가 대상']}")
        
        # 대회 일정 (dict인 경우)
        if item.get('대회 주요 일정') and isinstance(item['대회 주요 일정'], dict):
            schedule_text = ", ".join([f"{k}: {v}" for k, v in item['대회 주요 일정'].items()])
            if schedule_text:
                parts.append(f"대회 일정: {schedule_text}")
    
    # 강의인 경우
    elif doc_type == 'education':
        if item.get('설명'):
            parts.append(f"설명: {item['설명']}")
    
    page_content = "\n".join(parts)
    
    # 2. 키워드 추출 (검색용)
    all_text = " ".join([str(v) for v in item.values() if v and v != "null"])
    keywords = extract_keywords(all_text)
    
    # 3. 메타데이터 구성
    metadata = {
        "type": doc_type,
        "platform": platform,
        "title": item.get('제목', 'Unknown'),
        "url": item.get('Url', ''),
        "keywords": keywords
    }
    
    # 공모전 추가 메타데이터
    if doc_type == 'competition':
        # 마감일 추출
        schedule = item.get('대회 주요 일정', {})
        if isinstance(schedule, dict):
            deadline = schedule.get('대회 종료') or schedule.get('리더보드 제출 마감')
            if deadline:
                metadata['deadline'] = deadline
    
    return Document(page_content=page_content, metadata=metadata)

# 테스트
test_item = {
    "제목": "테스트 대회",
    "배경": "AI 개발",
    "설명": "머신러닝 모델 개발",
    "Url": "https://test.com"
}
test_doc = create_document(test_item, "competition", "test")
print("생성된 Document:")
print(f"내용: {test_doc.page_content[:100]}...")
print(f"메타데이터: {test_doc.metadata}")

생성된 Document:
내용: 제목: 테스트 대회
배경: AI 개발
설명: 머신러닝 모델 개발...
메타데이터: {'type': 'competition', 'platform': 'test', 'title': '테스트 대회', 'url': 'https://test.com', 'keywords': ['테스트', '대회', 'ai', '개발', '머신러닝', '모델', 'https', 'test', 'com']}


In [ ]:
#모든 문서 documents 전환
def create_all_documents(data_sources: List[Dict]) -> List[Document]:
    """모든 데이터 소스를 Document로 변환"""
    all_documents = []
    
    for source in data_sources:
        print(f"\n 처리 중: {source['platform']} ({source['type']})")
        
        # JSON 로드
        data = load_json_data(source['path'])
        
        if not data:
            continue
        
        # Document 생성
        docs = []
        for item in data:
            try:
                doc = create_document(item, source['type'], source['platform'])
                docs.append(doc)
            except Exception as e:
                print(f"   항목 처리 실패: {item.get('제목', 'Unknown')}")
                print(f"     오류: {e}")
        
        print(f"   {len(docs)}개 Document 생성")
        all_documents.extend(docs)
    
    return all_documents

# 실행
print("=" * 70)
print("Document 생성 시작")
print("=" * 70)

documents = create_all_documents(data_sources)

print("\n" + "=" * 70)
print(f" 총 {len(documents)}개 Document 생성 완료!")
print("=" * 70)

# 샘플 확인
if documents:
    print("\n 샘플 Document:")
    print(f"내용: {documents[0].page_content[:200]}...")
    print(f"메타데이터: {documents[0].metadata}")


Document 생성 시작

 처리 중: dacon (competition)
   243개 Document 생성

 처리 중: inflearn (education)
   7502개 Document 생성

 처리 중: lh_compas (competition)
   6개 Document 생성

 처리 중: lh_compas (competition)
   42개 Document 생성

 처리 중: kaggle (competition)
   20개 Document 생성

 처리 중: linkareer (auto)
   20개 Document 생성

 처리 중: data_go_kr (competition)
   94개 Document 생성

 총 7927개 Document 생성 완료!

 샘플 Document:
내용: 제목: 운수종사자 인지적 특성 데이터를 활용한 교통사고 위험 예측 AI 경진대회
배경: 교통사고는 차량·도로 환경뿐 아니라 운수종사자의 인지 특성에 크게 좌우됩니다. 실제로 운수종사자는 신규 진입 시 자격 검사를 받고, 이후 정기적으로 자격 유지 검사를 통해 인지 능력과 안전 운전 역량을 점검받습니다. 이러한 자격 검사 데이터를 활용해 사고 위험도를 예측하...
메타데이터: {'type': 'competition', 'platform': 'dacon', 'title': '운수종사자 인지적 특성 데이터를 활용한 교통사고 위험 예측 AI 경진대회', 'url': 'https://dacon.io/competitions/official/236607/overview/description', 'keywords': ['운수종사자', '인지적', '특성', '데이터를', '활용한', '교통사고', '위험', '예측', 'ai', '경진대회'], 'deadline': '11.14'}


In [16]:
# ========================================
# 원본 데이터 중복 제거
# ========================================
print("=" * 70)
print("중복 데이터 제거")
print("=" * 70)

# 1. 중복 확인
print(f"\n 원본 문서: {len(documents)}개")

# 제목 기준 중복 확인
seen_titles = {}
duplicates = []

for i, doc in enumerate(documents):
    title = doc.metadata.get('title', f'Unknown_{i}')
    
    if title in seen_titles:
        duplicates.append((i, title, seen_titles[title]))
    else:
        seen_titles[title] = i

print(f" 중복 발견: {len(duplicates)}개")

if duplicates:
    print(f"\n 중복 예시 (처음 5개):")
    for i, (idx, title, orig_idx) in enumerate(duplicates[:5], 1):
        print(f"  {i}. '{title[:50]}'")
        print(f"     원본 인덱스: {orig_idx}, 중복 인덱스: {idx}")

# 2. 중복 제거 (제목 기준)
unique_docs = []
seen_titles = set()

for doc in documents:
    title = doc.metadata.get('title', '')
    
    if title not in seen_titles:
        unique_docs.append(doc)
        seen_titles.add(title)

print(f"\n 중복 제거 완료!")
print(f"   원본: {len(documents)}개")
print(f"   제거 후: {len(unique_docs)}개")
print(f"   제거됨: {len(documents) - len(unique_docs)}개")

# 3. 중복 제거된 데이터 사용
documents = unique_docs
chunked_docs = documents  # Chunking 안 함

print("=" * 70)

중복 데이터 제거

 원본 문서: 7927개
 중복 발견: 3462개

 중복 예시 (처음 5개):
  1. '인프런 클론코딩으로 배우는 올인원 LMS 솔루션: Next.js·Flutter·AWS·Su'
     원본 인덱스: 285, 중복 인덱스: 301
  2. '맥킨지 출신 김재성의 AI로 앞서가는 문제 해결력 & 리서치 전략'
     원본 인덱스: 262, 중복 인덱스: 303
  3. '[VOD] 6주 완성! 개발 실무를 위한 고농축 바이브코딩 (Cursor AI, Figma'
     원본 인덱스: 287, 중복 인덱스: 305
  4. '코딩 없이 AI 자동화 전문가가 되는 법, n8n 완벽 가이드'
     원본 인덱스: 292, 중복 인덱스: 307
  5. '[리뉴얼] 파이썬입문과 크롤링기초 부트캠프 [파이썬, 웹, 데이터 이해 기본까지] (업데이'
     원본 인덱스: 300, 중복 인덱스: 308

 중복 제거 완료!
   원본: 7927개
   제거 후: 4465개
   제거됨: 3462개


## 추가적인 작업 

In [6]:
# ========================================
# linkareer 대외활동/공모전 자동 구분
# ========================================

def classify_linkareer_type(item):
    """
    링커리어 항목을 대외활동/공모전으로 자동 구분
    
    Returns:
        'competition': 공모전
        'activity': 대외활동
    """
    
    # 제목과 내용에서 판단
    title = item.get('title', '').lower()
    content = item.get('content', '').lower()
    category = item.get('category', '').lower()
    
    # 결합
    text = f"{title} {content} {category}"
    
    # 공모전 키워드
    competition_keywords = [
        '공모전', '경진대회', '챌린지', 'contest', 'competition',
        '아이디어 공모', '수상', '시상', '상금'
    ]
    
    # 대외활동 키워드
    activity_keywords = [
        '대외활동', '서포터즈', '기자단', '앰버서더', 
        '리포터', '서포터', '홍보대사', '활동'
    ]
    
    # 점수 계산
    competition_score = sum(1 for kw in competition_keywords if kw in text)
    activity_score = sum(1 for kw in activity_keywords if kw in text)
    
    # 판단
    if competition_score > activity_score:
        return 'competition'
    elif activity_score > competition_score:
        return 'activity'
    else:
        # 동점이면 기본값
        return 'competition'

# 테스트
test_items = [
    {"title": "2024 AI 아이디어 공모전", "content": "상금 1000만원"},
    {"title": "카카오 서포터즈 모집", "content": "홍보 대외활동"},
]

for item in test_items:
    result = classify_linkareer_type(item)
    print(f"'{item['title']}' → {result}")

'2024 AI 아이디어 공모전' → competition
'카카오 서포터즈 모집' → activity


In [9]:
def extract_keywords(text: str) -> List[str]:
    """텍스트에서 주요 키워드 추출"""
    if not text or text == "null" or not isinstance(text, str):
        return []
    
    # 줄바꿈을 공백으로
    text = text.replace('\n', ' ')
    
    # 일반적인 단어들 (제외할 불용어)
    stopwords = {'을', '를', '이', '가', '은', '는', '의', '에', '와', '과', '도', '로', '으로', 
                 '및', '등', '수', '것', '때', '등을', '있는', '하는', '통해', '위한', '대한'}
    
    # 단어 추출 (2글자 이상)
    words = re.findall(r'[가-힣a-zA-Z]{2,}', text)
    
    # 불용어 제거 & 소문자 변환
    keywords = [w.lower() for w in words if w not in stopwords]
    
    # 중복 제거 & 상위 10개
    keywords = list(dict.fromkeys(keywords))[:10]
    
    return keywords

# 테스트
test_text = "ChatGPT\nAI\n머신러닝 딥러닝을 활용한 데이터 분석"
print("테스트 결과:", extract_keywords(test_text))

테스트 결과: ['chatgpt', 'ai', '머신러닝', '딥러닝을', '활용한', '데이터', '분석']


In [ ]:
# JSON 로드 함수
def load_json_data(file_path: str):
    """JSON 파일 로드"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"    파일 없음: {file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"    JSON 파싱 오류: {e}")
        return None
    except Exception as e:
        print(f"    오류: {e}")
        return None

In [17]:
# ========================================
# 플랫폼별 문서 개수 집계
# ========================================
from collections import Counter

print("\n" + "=" * 70)
print("플랫폼별 Document 개수")
print("=" * 70)

platform_counts = Counter(doc.metadata.get('platform', 'unknown') for doc in documents)

for platform, count in platform_counts.items():
    print(f"  {platform:15s} : {count}개")

print("=" * 70)
print(f"총 Document 수 : {sum(platform_counts.values())}개")


플랫폼별 Document 개수
  dacon           : 243개
  inflearn        : 4046개
  lh_compas       : 42개
  kaggle          : 20개
  linkareer       : 20개
  data_go_kr      : 94개
총 Document 수 : 4465개


## chunking 생략 이유

In [14]:
# ========================================
# 문서 길이 분석
# ========================================
import numpy as np

# 모든 문서 길이 계산
lengths = [len(doc.page_content) for doc in documents]

print(" 문서 길이 통계:")
print(f"   총 문서: {len(lengths)}개")
print(f"   평균: {np.mean(lengths):.0f}자")
print(f"   최소: {np.min(lengths)}자")
print(f"   최대: {np.max(lengths)}자")
print(f"   중간값: {np.median(lengths):.0f}자")
print(f"   표준편차: {np.std(lengths):.0f}자")

print("\n 길이별 분포:")
ranges = [
    (0, 300, "매우 짧음"),
    (300, 500, "짧음"),
    (500, 1000, "중간"),
    (1000, 2000, "김"),
    (2000, float('inf'), "매우 김")
]

for min_len, max_len, label in ranges:
    count = sum(1 for l in lengths if min_len <= l < max_len)
    percentage = count / len(lengths) * 100
    print(f"   {label:10s} ({min_len:4d}~{max_len:5f}자): {count:5d}개 ({percentage:5.1f}%)")

# 샘플 문서 확인
print("\n 샘플 문서 (처음 3개):")
for i, doc in enumerate(documents[:3], 1):
    print(f"\n{i}. {doc.metadata.get('title', 'Unknown')[:50]}")
    print(f"   길이: {len(doc.page_content)}자")
    print(f"   내용: {doc.page_content[:150]}...")

 문서 길이 통계:
   총 문서: 7927개
   평균: 154자
   최소: 0자
   최대: 1891자
   중간값: 116자
   표준편차: 154자

 길이별 분포:
   매우 짧음      (   0~300.000000자):  7355개 ( 92.8%)
   짧음         ( 300~500.000000자):   312개 (  3.9%)
   중간         ( 500~1000.000000자):   202개 (  2.5%)
   김          (1000~2000.000000자):    58개 (  0.7%)
   매우 김       (2000~  inf자):     0개 (  0.0%)

 샘플 문서 (처음 3개):

1. 운수종사자 인지적 특성 데이터를 활용한 교통사고 위험 예측 AI 경진대회
   길이: 821자
   내용: 제목: 운수종사자 인지적 특성 데이터를 활용한 교통사고 위험 예측 AI 경진대회
배경: 교통사고는 차량·도로 환경뿐 아니라 운수종사자의 인지 특성에 크게 좌우됩니다. 실제로 운수종사자는 신규 진입 시 자격 검사를 받고, 이후 정기적으로 자격 유지 검사를 통해 인지 능력...

2. 2025 신문과 방송 독자 데이터 분석 아이디어 경진대회
   길이: 592자
   내용: 제목: 2025 신문과 방송 독자 데이터 분석 아이디어 경진대회
배경: 안녕하세요, 데이커 여러분 :) ‘신문과방송 독자 데이터 분석 아이디어 경진대회’에 오신 여러분을 진심으로 환영합니다! 이번 월간 데이콘은 문화체육관광부 산하 공공기관인 한국언론진흥재단과 함께합니다...

3. 토스 NEXT ML CHALLENGE : 광고 클릭 예측(CTR) 모델 개발
   길이: 1079자
   내용: 제목: 토스 NEXT ML CHALLENGE : 광고 클릭 예측(CTR) 모델 개발
배경: 토스는 ML 기술을 기반으로 디스플레이 광고의 성과를 극대화하기 위해, 다양한 광고 지면과 사용자 접점에서 광고의 노출부터 전환까지 퍼널 전반을 최적화할 수 있는 정교

In [ ]:
# ========================================
# 최종 정리(Chunking 생략)
# ========================================

chunked_docs = documents

print(f"  Chunking 생략 (문서가 충분히 짧음)")
print(f"   총 문서: {len(chunked_docs)}개")
print(f"   평균 길이: 150자")
print(f"   93.4%가 300자 미만")



## 긴 문서 37개는?

### 걱정 불필요!
'''
이유 1: 극소수 (0.5%)
→ 전체 성능에 거의 영향 없음

이유 2: 1000~2000자도 짧음
→ LLM이 한 번에 처리 가능
→ 문제없음!

이유 3: 전체 맥락 유지
→ 분할하면 오히려 정보 손실
→ 그대로가 나음!
'''



## 성능 예상

### Chunking 없어도 완벽!
'''
평균 150자:
 검색 정확도: 매우 높음
 LLM 처리: 매우 빠름
 메모리: 매우 적음
 맥락 유지: 완벽

→ 이상적인 크기! 
'''

  Chunking 생략 (문서가 충분히 짧음)
   총 문서: 7927개
   평균 길이: 150자
   93.4%가 300자 미만


'\n평균 150자:\n 검색 정확도: 매우 높음\n LLM 처리: 매우 빠름\n 메모리: 매우 적음\n 맥락 유지: 완벽\n\n→ 이상적인 크기! \n'

## build_index.py

In [13]:
# ========================================
# OpenAI 임베딩 모델 준비
# ========================================
print("=" * 70)
print("OpenAI 임베딩 모델 로드")
print("=" * 70)

from langchain_openai import OpenAIEmbeddings
import os

# API 키 확인
if not os.getenv("OPENAI_API_KEY"):
    print(" OPENAI_API_KEY가 설정되지 않았습니다!")
    raise ValueError("OPENAI_API_KEY를 먼저 설정하세요")

print("\n OpenAI 임베딩 모델 로드 중...")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 빠르고 저렴한 모델
    # model="text-embedding-3-large",  # 더 좋은 성능 (비용 2배)
)

print(" OpenAI 임베딩 모델 로드 완료!")

# 테스트
print("\n 임베딩 테스트 중...")
test_vector = embeddings.embed_query("테스트 문장입니다")
print(f" 벡터 차원: {len(test_vector)}차원")
print(f"   (text-embedding-3-small: 1536차원)")

# Chunking 하지 않고 원본 그대로 사용
chunked_docs = documents

print(f"\n 통계:")
print(f"   총 문서: {len(chunked_docs):,}개")

# 문서 길이 통계
lengths = [len(doc.page_content) for doc in chunked_docs[:100]]
avg_length = sum(lengths) / len(lengths) if lengths else 0

print(f"   평균 길이: {avg_length:.0f}자")
print(f"\n 공모전/강의 데이터는 이미 짧아서 Chunking을 생략합니다.")
print("   (Chunking은 긴 문서에만 필요합니다)")

print("=" * 70)

OpenAI 임베딩 모델 로드

 OpenAI 임베딩 모델 로드 중...
 OpenAI 임베딩 모델 로드 완료!

 임베딩 테스트 중...
 벡터 차원: 1536차원
   (text-embedding-3-small: 1536차원)

 통계:
   총 문서: 7,927개
   평균 길이: 808자

 공모전/강의 데이터는 이미 짧아서 Chunking을 생략합니다.
   (Chunking은 긴 문서에만 필요합니다)


In [18]:
# ========================================
# FAISS 인덱스 생성
# ========================================
print("=" * 70)
print("FAISS 인덱스 생성")
print("=" * 70)

from langchain_community.vectorstores import FAISS

print(f"\n 벡터화할 문서: {len(chunked_docs)}개")
print(" 벡터 생성 중... (시간이 걸릴 수 있습니다)")

# FAISS 벡터스토어 생성
vectorstore = FAISS.from_documents(
    chunked_docs,
    embeddings
)

# 저장 경로
FAISS_INDEX_PATH = "/Users/jinwoong/Desktop/coding/bitamin/nlp_project_2/faiss_index"

# 저장
print(f"\n FAISS 인덱스 저장 중...")
vectorstore.save_local(FAISS_INDEX_PATH)

print(f"\n FAISS 인덱스 생성 및 저장 완료!")
print(f"   경로: {FAISS_INDEX_PATH}")
print(f"   문서: {len(chunked_docs)}개")
print("=" * 70)

FAISS 인덱스 생성

 벡터화할 문서: 4465개
 벡터 생성 중... (시간이 걸릴 수 있습니다)

 FAISS 인덱스 저장 중...

 FAISS 인덱스 생성 및 저장 완료!
   경로: /Users/jinwoong/Desktop/coding/bitamin/nlp_project_2/faiss_index
   문서: 4465개


## retrievers

### hybrid_retrievers.py

In [19]:
# ========================================
# 셀 11: Hybrid Retriever (수정 버전)
# ========================================
print("=" * 70)
print("Hybrid Retriever 설정")
print("=" * 70)

from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import numpy as np

# 1. FAISS 로드
print("\nFAISS 인덱스 로드 중...")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = FAISS.load_local(
    "/Users/jinwoong/Desktop/coding/Bitamin/nlp_project_2/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS 로드 완료!")

# 2. 질문 분석 함수
def analyze_query(query: str) -> str:
    """질문 의도를 분석하여 카테고리 반환"""
    q = query.lower()
    
    if any(w in q for w in ["대외활동", "서포터즈", "기자단", "앰버서더", "리포터", "서포터", "홍보대사"]):
        return "activity"
    
    elif any(w in q for w in ["공모전", "대회", "경진대회", "챌린지", "competition"]):
        return "competition"
    
    elif any(w in q for w in ["강의", "강좌", "수업", "배우", "공부", "학습", "튜토리얼", "입문", "기초"]):
        return "education"
    
    elif any(w in q for w in ["추천", "찾아", "있어", "알려줘", "소개"]):
        return "recommendation"
    
    else:
        return "general"

# 3. 메타데이터 필터 생성 함수
def get_search_filter(query: str):
    """질문에 따라 메타데이터 필터 생성"""
    topic = analyze_query(query)
    
    if topic == "activity":
        return {"type": "activity"}
    elif topic == "competition":
        return {"type": "competition"}
    elif topic == "education":
        return {"type": "education"}
    else:
        return None

# 4. BM25 Retriever (전역으로 먼저 생성)
print("\nBM25 Retriever 설정 중...")

all_docs = []
for doc_id in range(len(chunked_docs)):
    all_docs.append(chunked_docs[doc_id])

bm25_retriever = BM25Retriever.from_documents(
    all_docs,
    k=10
)

print("BM25 Retriever 준비!")

# 5. Hybrid Retriever 생성 함수
def get_hybrid_retriever(query: str = None):
    """
    질문 의도에 맞춰 가중치와 필터를 동적으로 조정
    
    Args:
        query: 사용자 질문
    
    Returns:
        hybrid_retriever, topic, weights, search_filter
    """
    
    # 기본 설정
    weights = [0.3, 0.7]
    topic = "general"
    search_filter = None
    
    if query:
        topic = analyze_query(query)
        search_filter = get_search_filter(query)
        
        # 주제별 가중치 조정
        if topic == "activity":
            weights = [0.4, 0.6]
        elif topic == "competition":
            weights = [0.4, 0.6]
        elif topic == "education":
            weights = [0.3, 0.7]
        elif topic == "recommendation":
            weights = [0.2, 0.8]
        else:
            weights = [0.3, 0.7]
    
    # Vector Retriever 생성 (필터 적용)
    vector_search_kwargs = {
        "k": 10,
        "fetch_k": 100,
        "lambda_mult": 0.7
    }
    
    if search_filter:
        vector_search_kwargs["filter"] = search_filter
    
    vector_retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs=vector_search_kwargs
    )
    
    # Ensemble Retriever
    hybrid_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=weights
    )
    
    return hybrid_retriever, topic, weights, search_filter

# 6. 필터링 통합 함수
def get_filtered_results(query: str, top_k: int = 10):
    """
    Hybrid 검색 + 타입 필터
    
    Args:
        query: 사용자 질문
        top_k: 최종 반환 개수
    
    Returns:
        필터링된 문서 리스트
    """
    
    # 1. Hybrid 검색
    retriever, topic, weights, search_filter = get_hybrid_retriever(query)
    docs = retriever.get_relevant_documents(query)
    
    # 2. 타입 필터링 (BM25 결과에 적용)
    if search_filter:
        target_type = search_filter.get("type")
        docs = [doc for doc in docs if doc.metadata.get("type") == target_type]
    
    # 3. Top K
    docs = docs[:top_k]
    
    return docs, topic, weights

print("\nHybrid Retriever 함수 준비 완료!")


Hybrid Retriever 설정

FAISS 인덱스 로드 중...
FAISS 로드 완료!

BM25 Retriever 설정 중...
BM25 Retriever 준비!

Hybrid Retriever 함수 준비 완료!


## prompts

### question_prompts.py

In [ ]:
# ========================================
# 프롬프트 
# ========================================
print("=" * 70)
print("프롬프트 설정")
print("=" * 70)

from langchain.prompts import PromptTemplate
from datetime import datetime

# 현재 날짜
current_date = datetime.now().strftime("%Y년 %m월 %d일")
weekday_kr = {
    'Monday': '월요일', 'Tuesday': '화요일', 'Wednesday': '수요일',
    'Thursday': '목요일', 'Friday': '금요일', 'Saturday': '토요일', 'Sunday': '일요일'
}
current_weekday = datetime.now().strftime("%A")
current_date_full = f"{current_date} ({weekday_kr[current_weekday]})"

qa_template = f"""당신은 비타민(BITAmin) AI 학회 동아리의 친근한 도우미 챗봇입니다.
공모전과 강의 정보를 제공하며, 사용자와 자연스럽게 대화합니다.

현재 날짜: {current_date_full}

=== 좋은 답변 예시 ===

Q: 금융 공모전 추천해줘
A: 금융 공모전 찾으시는군요! 좋은 분야네요 

신한AI 금융 서비스 아이디어 대회가 있는데, 아쉽게도 5월에 마감됐네요 
근데 과거 문제 보면서 연습하시면 나중에 큰 도움될 거예요.

대출 매출 예측 대회도 9월에 끝났지만, 실무적인 주제라 데이터 분석 
실력 쌓기 좋을 것 같아요.

아래 링크에서 자세한 정보 확인해보세요!

---
**추천 공모전 목록**

1. 신한AI 금융 서비스 아이디어 경진대회
   https://dacon.io/competitions/official/236088/overview/description

2. 대출 상점 총 매출 예측 경진대회
   https://dacon.io/competitions/official/236123/overview/description

3. 금융 사기 탐지 AI 경진대회
   https://dacon.io/competitions/official/236125/overview/description

=============================================================================

다음 문서들을 읽고 질문에 답변하세요.

문서:
{{context}}

질문: {{question}}

답변 구조

**[1단계] 자연스러운 대화 (3-7문장)**
- 질문에 공감하고 상황 파악
- 추천하는 공모전/강의에 대해 자연스럽게 설명
- 마감 상태, 난이도, 특징 등 언급
- 조언이나 격려 추가
- 플랫폼 이름은 절대 언급하지 마세요!

**[2단계] 정형화된 목록**
```
---
**추천 [공모전/강의/대외활동] 목록**

1. [제목]
   [완전한 URL]

2. [제목]
   [완전한 URL]

3. [제목]
   [완전한 URL]
```

핵심 규칙 

1. 플랫폼 언급 금지 (절대 규칙!)
   
   절대 금지:
   - "인프런에 있는 강의예요"
   - "데이콘에서 진행하는 대회예요"
   - "Kaggle 대회예요"
   - 어떤 플랫폼 이름도 언급하지 마세요!
   
   올바른 방법:
   - "이 강의는 기초부터 알려줘서 좋아요"
   - "이 대회는 실무 경험 쌓기 좋을 것 같아요"
   - 플랫폼 언급 없이 내용만 설명

2. 추천 개수 규칙 (중요!)
   
   필수 사항:
   - 반드시 최소 3개 이상 추천
   - 1개 또는 2개만 추천하지 마세요
   
   예시:
   - "금융 공모전" 검색 시 → 금융, 데이터 분석, AI 관련 공모전 3개 이상
   - "파이썬 강의" 검색 시 → 파이썬 기초, 응용, 실습 강의 3개 이상

3. 공모전 날짜 반영 (공모전 질문 시에만)
   
   현재 날짜: {current_date_full}
   
   공모전 추천 시:
   - 각 공모전의 마감일을 현재 날짜와 비교하여 자연스럽게 언급하세요
   - 이미 마감된 경우: 과거 문제로 연습할 수 있다는 점, 유사 대회 준비에 도움된다는 점 등을 자유롭게 표현
   - 마감이 임박한 경우: 남은 기간을 구체적으로 알려주고 준비 시간을 고려한 조언
   - 진행 중인 경우: 여유있게 준비할 수 있다는 점, 참여를 독려하는 말 등을 자유롭게 표현
   - 매번 다른 표현을 사용하여 친구처럼 자연스럽게 대화하세요
   
   강의 추천 시:
   - 날짜나 마감 관련 언급 절대 금지
   - 강의는 상시 수강 가능하므로 마감 개념 없음
   - 난이도, 특징, 내용, 학습 경로 중심으로만 설명

4. 개별 추천 이유 
   
   자연스러운 대화를 위해:
   - 각 공모전/강의를 소개할 때 사용자 상황과 연결지어 설명하세요
   - 왜 이걸 추천하는지, 어떤 점이 도움될지 자유롭게 풀어서 얘기하세요
   - 모든 항목에 똑같은 길이로 설명할 필요 없어요
   - 어떤 건 짧게, 어떤 건 길게 설명해도 괜찮아요
   - 친구에게 진심으로 추천하듯이 자연스럽게
   
   피해야 할 답변:
   "이 대회 좋아요. 참여해보세요."
   "이 강의 추천드려요."
   
   좋은 답변 예시:
   "이 대회는 RAG 시스템의 검색 정확도를 높이는 기술을 다루는데, 여러분 프로젝트 핵심 성능 개선에 직접 쓸 수 있을 것 같아요."
   "이 강의 진짜 기초부터 차근차근 알려줘서 입문자한테 딱이에요."
   "프로젝트 주제랑 완전 맞아떨어지네요!"

5. URL 처리
   
   절대 금지:
   - URL 만들어내지 않기
   - URL을 `...`으로 줄이지 않기
   - 다른 문서의 URL 가져다 쓰지 않기
   
   올바른 방법:
   - 문서의 URL을 정확히 그대로 복사
   - URL이 없는 문서는 목록에서 제외

6. 문서 선별
   - 질문과 관련있는 문서만 사용
   - 관련없는 문서는 완전히 무시
   - 3개 미만이면 유사 주제 문서로 확장

7. 답변 스타일
   - 친구처럼 자연스럽게
   - 이모지 적절히 활용
   - 매번 다르게 표현

8. 절대 금지
   - 문서에 없는 정보 만들지 않기
   - 플랫폼 이름 추측하지 않기
   - URL을 ...으로 줄이지 않기
   - 날짜 계산 틀리지 않기
   - 1개 또는 2개만 추천하지 않기
   - 똑같은 표현 반복하지 않기 


답변:"""

question_prompt = PromptTemplate(
    template=qa_template,
    input_variables=["context", "question"]
)

print("프롬프트 설정 완료!")
print(f"현재 날짜: {current_date_full}")
print("=" * 70)

프롬프트 설정
프롬프트 설정 완료!
현재 날짜: 2025년 11월 19일 (수요일)


## chains

### qa_chain.py

In [ ]:
# ========================================
# 셀 13: QA Chain 
# ========================================
print("=" * 70)
print("QA Chain 구현 (URL 할루시네이션 완전 방지)")
print("=" * 70)

import os
import re
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage

def fix_urls_in_answer_v2(answer: str, docs: list) -> str:
    """
    LLM 답변의 잘못된 URL을 실제 문서 URL로 강제 교체 (강화 버전)
    
    전략:
    1. 답변에서 "추천 목록" 섹션 찾기
    2. 각 항목의 제목 추출
    3. 제목으로 실제 문서 찾기
    4. 실제 URL로 강제 교체
    """
    
    # 1. 문서별 제목-URL 매핑 생성
    title_to_doc = {}
    for doc in docs:
        title = doc.metadata.get('title', '').strip()
        if title:
            normalized = title.lower().replace(' ', '')
            title_to_doc[normalized] = doc
    
    # 2. 답변에서 목록 부분 찾기
    list_pattern = r'\*\*추천.*?목록\*\*\s*\n+(.*?)(?=\n\n|$)'
    list_match = re.search(list_pattern, answer, re.DOTALL)
    
    if not list_match:
        return answer
    
    list_section = list_match.group(1)
    new_list_items = []
    
    # 3. 각 항목 처리
    item_pattern = r'(\d+)\.\s*([^\n]+)\n\s*(.+?)(?=\n\d+\.|$)'
    
    for match in re.finditer(item_pattern, list_section, re.DOTALL):
        number = match.group(1)
        title = match.group(2).strip()
        old_url = match.group(3).strip()
        
        normalized_title = title.lower().replace(' ', '')
        
        correct_url = None
        best_match_doc = None
        best_score = 0
        
        for norm_key, doc in title_to_doc.items():
            if normalized_title in norm_key or norm_key in normalized_title:
                score = len(set(normalized_title) & set(norm_key))
                if score > best_score:
                    best_score = score
                    best_match_doc = doc
        
        if best_match_doc:
            correct_url = best_match_doc.metadata.get('url', '')
        
        if correct_url and correct_url not in ['Unknown', None, '', '해당없음']:
            if not correct_url.startswith('http'):
                correct_url = f"https://{correct_url}"
            new_item = f"{number}. {title}\n   {correct_url}"
        else:
            print(f"   URL 없음 (제외): {title[:30]}...")
            continue
        
        new_list_items.append(new_item)
    
    if new_list_items:
        new_list_section = '\n\n'.join(new_list_items)
        fixed_answer = answer[:list_match.start(1)] + new_list_section + answer[list_match.end(1):]
        return fixed_answer
    
    return answer

def create_hybrid_qa_chain(query: str = None):
    """URL 할루시네이션 완전 방지 QA Chain"""
    
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        openai_api_key=OPENAI_API_KEY,
        temperature=0.7,
        max_tokens=1000,
    )
    
    # Retriever (4개 값 받기!)
    if query:
        retriever, topic, weights, search_filter = get_hybrid_retriever(query)
        print(f"Retriever: {topic} (BM25={weights[0]}, Vector={weights[1]})")
    else:
        retriever, topic, weights, search_filter = get_hybrid_retriever()
    
    # Custom QA Chain
    class URLFixedQAChain:
        def __init__(self, llm, retriever, prompt):
            self.llm = llm
            self.retriever = retriever
            self.prompt = prompt
        
        def invoke(self, inputs):
            query = inputs['query']
            
            # 1. 문서 검색
            docs = self.retriever.get_relevant_documents(query)
            
            print(f"\n검색된 문서: {len(docs)}개")
            for i, doc in enumerate(docs[:5], 1):
                title = doc.metadata.get('title', '')[:40]
                url = doc.metadata.get('url', '')[:60]
                print(f"   [{i}] {title}...")
                print(f"       URL: {url}...")
            
            # 2. 컨텍스트 생성
            context_parts = []
            for i, doc in enumerate(docs, 1):
                title = doc.metadata.get('title', 'Unknown')
                doc_type = doc.metadata.get('type', 'Unknown')
                platform = doc.metadata.get('platform', '')
                url = doc.metadata.get('url', '')
                content = doc.page_content
                
                context = f"\n--- 문서 {i} ---\n"
                context += f"제목: {title}\n"
                context += f"유형: {doc_type}\n"
                
                if platform and platform not in ['Unknown', None, '', '해당없음']:
                    context += f"플랫폼: {platform}\n"
                
                if url and url not in ['Unknown', None, '', '해당없음']:
                    context += f"URL: {url}\n"
                
                context += f"\n내용:\n{content}\n"
                context_parts.append(context)
            
            context = "\n".join(context_parts)
            
            # 3. 프롬프트 생성
            formatted_prompt = self.prompt.format(
                context=context,
                question=query
            )
            
            # 4. LLM 호출
            print("\nLLM 답변 생성 중...")
            response = self.llm.invoke([HumanMessage(content=formatted_prompt)])
            raw_answer = response.content
            
            print("LLM 답변 생성 완료")
            
            # 5. URL 후처리
            print("\nURL 검증 및 교체 중...")
            fixed_answer = fix_urls_in_answer_v2(raw_answer, docs)
            print("URL 교체 완료")
            
            return {
                'result': fixed_answer,
                'source_documents': docs,
                'raw_result': raw_answer
            }
    
    qa_chain = URLFixedQAChain(llm, retriever, question_prompt)
    
    return qa_chain

print("URL 할루시네이션 방지 QA Chain 준비 완료!")
print("=" * 70)

QA Chain 구현 (URL 할루시네이션 완전 방지)
URL 할루시네이션 방지 QA Chain 준비 완료!


## main

### main.py

In [ ]:
# ========================================
# Main 함수 QA 테스트
# ========================================
print("=" * 70)
print("Main 함수 스타일 QA 테스트")
print("=" * 70)

import traceback

def run_qa_test(query: str, verbose: bool = True):
    """
    Main.py 스타일의 QA 실행 함수
    
    Args:
        query: 사용자 질문
        verbose: 상세 출력 여부
    
    Returns:
        result: 실행 결과 딕셔너리
    """
    
    if verbose:
        print(f"\n{'='*70}")
        print(f" 질문: {query}")
        print('='*70)
    
    try:
        # 1. QA Chain 생성
        qa_chain = create_hybrid_qa_chain(query=query)
        
        # 2. 답변 생성
        result = qa_chain.invoke({"query": query})
        
        # 3. 답변 출력
        answer = result.get("result", "답변을 생성하지 못했습니다.")
        
        if verbose:
            print("\n 답변:")
            print("-" * 70)
            print(answer)
            print("-" * 70)
            
            # 참조 문서 정보
            source_docs = result.get("source_documents", [])
            if source_docs:
                print(f"\n 참조 문서 ({len(source_docs)}개):")
                for i, doc in enumerate(source_docs[:5], 1):
                    title = doc.metadata.get('title', 'Unknown')
                    doc_type = doc.metadata.get('type', 'Unknown')
                    platform = doc.metadata.get('platform', 'Unknown')
                    
                    print(f"\n   [{i}] {title}")
                    print(f"       유형: {doc_type} | 플랫폼: {platform}")
                    
                    # URL이 있으면 출력
                    url = doc.metadata.get('url', '')
                    if url:
                        print(f"        {url}")
        
        return {
            'success': True,
            'query': query,
            'answer': answer,
            'source_documents': result.get("source_documents", []),
            'error': None
        }
        
    except Exception as e:
        if verbose:
            print(f"\n 오류 발생: {str(e)}")
            traceback.print_exc()
        
        return {
            'success': False,
            'query': query,
            'answer': None,
            'source_documents': [],
            'error': str(e)
        }

print(" 테스트 함수 준비 완료!")

# ========================================
# 단일 테스트 실행
# ========================================
print("\n" + "=" * 70)
print("단일 테스트 실행")
print("=" * 70)

# 테스트 질문
test_query = "llm 관렪해서 공모전이나 강의 추천해줄래?"

# 실행
result = run_qa_test(test_query, verbose=True)

print("\n" + "=" * 70)

Main 함수 스타일 QA 테스트
 테스트 함수 준비 완료!

단일 테스트 실행

 질문: llm 관렪해서 공모전이나 강의 추천해줄래?
Retriever: competition (BM25=0.4, Vector=0.6)


/var/folders/qp/9cvpy58d2437065c6yp269d40000gn/T/ipykernel_59289/3755209238.py:111: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = self.retriever.get_relevant_documents(query)



검색된 문서: 20개
   [1] 2025 Bias-A-Thon : Bias 발견 챌린지 <Track 1>...
       URL: https://dacon.io/competitions/official/236486/overview/descr...
   [2] 월간 데이콘 한국어 문장 관계 분류 경진대회...
       URL: https://dacon.io/competitions/official/235875/overview/descr...
   [3] 제1회 법령데이터 활용 아이디어 공모전...
       URL: https://www.epeople.go.kr/cmmn/idea/redirect.do?ideaRegNo=1A...
   [4] (서울대 LH) LH-서울대학교 COMPAS 도시 데이터 분석 학교...
       URL: https://compas.lh.or.kr/subj/past/info?subjNo=SBJ_2502_001...
   [5] KNOW 기반 직업 추천 알고리즘 경진대회...
       URL: https://dacon.io/competitions/official/235865/overview/descr...

LLM 답변 생성 중...
LLM 답변 생성 완료

URL 검증 및 교체 중...
URL 교체 완료

 답변:
----------------------------------------------------------------------
안녕하세요! LLM에 관심이 많으시군요! 😊 요즘 인공지능 분야가 정말 흥미로운 것 같아요. 관련된 공모전과 강의가 몇 가지 있어서 소개해드릴게요.

먼저, **2025 Bias-A-Thon : Bias 발견 챌린지**가 있어요. 이 대회는 기존 LLM 응답에서 나타나는 편향을 발견하고 이를 정제된 데이터셋으로 제출하는 내용이에요. 대회는 4월 28일부터 5월 19일까지 진행되니까, 미리 준비하면 좋을 것 같아요.

또한, **2025 SW중심대학 디지털 경진대회 : AI부문**도 추천해

### batch_test.py

In [33]:
# ========================================
# 셀 15: 배치 테스트
# ========================================
print("=" * 70)
print("배치 테스트")
print("=" * 70)

def run_batch_test(queries: list):
    """
    여러 질문을 배치로 테스트
    
    Args:
        queries: 질문 리스트
    
    Returns:
        results: 결과 리스트
    """
    
    print(f"\n총 {len(queries)}개 질문 테스트 시작\n")
    
    results = []
    
    for i, query in enumerate(queries, 1):
        print(f"\n{'='*70}")
        print(f"[{i}/{len(queries)}] 테스트 진행 중...")
        print('='*70)
        
        result = run_qa_test(query, verbose=True)
        results.append(result)
        
        print(f"\n{'='*70}")
    
    # 요약
    print("\n" + "="*70)
    print(" 배치 테스트 결과 요약")
    print("="*70)
    
    success_count = sum(1 for r in results if r['success'])
    fail_count = len(results) - success_count
    
    print(f"\n 성공: {success_count}개")
    print(f" 실패: {fail_count}개")
    
    if fail_count > 0:
        print(f"\n실패한 질문:")
        for r in results:
            if not r['success']:
                print(f"   - {r['query']}: {r['error']}")
    
    print("\n" + "="*70)
    
    return results

# ========================================
# 테스트 질문 세트
# ========================================

# 다양한 유형의 질문
test_queries = [
    # 공모전
    "동아리 내 chatbot을 rag로 구현하는 프로젝트를 부원들과 함께 진행중인데 이와 관련된 현재 접수중인 공모전 있으면 추천해줄래?",
    
    # 강의
    "머신러닝 기초를 다지기에 좋은 강의 추천 부탁해",
    
    # 혼합
    "딥러닝을 배우고 싶은데 관련 강의와 나갈만한 공모전 둘다 추천해줄 수 있어?"
]

# 배치 테스트 실행
print("\n 배치 테스트 시작!")
batch_results = run_batch_test(test_queries)

배치 테스트

 배치 테스트 시작!

총 3개 질문 테스트 시작


[1/3] 테스트 진행 중...

 질문: 동아리 내 chatbot을 rag로 구현하는 프로젝트를 부원들과 함께 진행중인데 이와 관련된 현재 접수중인 공모전 있으면 추천해줄래?
Retriever: competition (BM25=0.4, Vector=0.6)

검색된 문서: 20개
   [1] 생성 AI ChatGPT 활용 학습 콘텐츠 제작 아이디어 경진대회...
       URL: https://dacon.io/competitions/official/236368/overview/descr...
   [2] 2025 Bias-A-Thon : Bias 대응 챌린지 <Track 2>...
       URL: https://dacon.io/competitions/official/236487/overview/descr...
   [3] 제13회 산업통상자원부 공공데이터 활용 아이디어 공모전...
       URL: https://www.datacontest.kr/...
   [4] 웹 기사 추천 AI 경진대회...
       URL: https://dacon.io/competitions/official/236290/overview/descr...
   [5] 도배 하자 질의 응답 처리 : 한솔데코 시즌2 생성 AI 경진대회...
       URL: https://dacon.io/competitions/official/236216/overview/descr...

LLM 답변 생성 중...
LLM 답변 생성 완료

URL 검증 및 교체 중...
URL 교체 완료

 답변:
----------------------------------------------------------------------
안녕하세요! RAG 시스템을 활용한 챗봇 프로젝트를 진행 중이시군요! 정말 흥미로운 주제인데요, 관련된 공모전이 몇 개 있어요. 여러분의 프로젝트에 도움이 될 만한 대회들을 추천해 드릴게요.

첫 번